# 00. Scope and methodology rationale

Why this dataset, why TVs and Samsung, and why the project proceeds to `01` and `02`.
This is the team's internal exploratory decision record, not final report prose.
It replaces the useful exploratory roles of `Cleaning_Description`,
`Shopper_Type_Analysis` and `both_brands`, and retains methodological lessons from
`Edoardo_notebook`. None of those files is required to read or execute this notebook.
The maintained analytical route is `00` → `01` → `02` → subsequent modelling.

Run from `Marketing-Project`. Only the two original CSVs are needed. This notebook
keeps the screening evidence that precedes the official build, without rebuilding
considerations, estimating gaps or fitting models. All displayed numbers are computed.

**For a reader who does not inspect code:** read Sections 1–6, the self-contained
Evidence record E1–E12, and the methodological handoff in Section 9. The expandable
aggregate tables provide detail. Python cells and the provenance map are optional.


## 1. The question this work now supports

**Primary RQ:** How much of LG's shortfall in observed TV purchases against Samsung
at this retailer is associated with fewer sessions reaching LG's models, with the
number of models considered per reaching session, and with lower purchase conversion?

**Supporting question:** For the conversion component, how does the LG–Samsung
comparison change after accounting for initial price, month and the shopper's
observed browsing situation before the model is opened?

This follows the recent reach/conversion direction in `02`, particularly Section 4.7,
and the revised report guide's Section 2. The earlier question concentrated on
view-to-cart and cart-to-purchase differences after price/month adjustment. The
exploration then raised a second issue: shoppers can arrive with different prior
browsing histories. The retained direction integrates that lesson while placing
conversion inside the broader shortfall in purchased considerations.

The decision maker is LG's team working with this retailer. The study can prioritise
an assortment, visibility or purchase-journey investigation and specify what to test
with the retailer. It cannot rank interventions by expected profit or establish that
placement, pages or checkout caused the observed patterns.

## 2. One role for each step

| Step | Question answered | Operations and outputs it owns |
| --- | --- | --- |
| **00: rationale** | Why is this scope defensible, what alternatives were considered, and what analysis follows? | Read-only inventory of the original sources; broader catalogue coverage; review feasibility; decision and method rationale. No derived datasets or project-result files. |
| **01: build** | What exactly is an observation, and how are its variables constructed? | Event cleaning; first-view considerations; timestamp rules; prior browsing and outcomes; top-model split; build parameters and audits; derived CSVs. |
| **02: preliminary analysis** | What patterns does the constructed dataset show? | Category gap comparison, TV price distributions, funnel rates, shopper mix, navigation, session outcomes, reach decomposition, concentration and uncertainty tables/figures. |
| **03: proposed models** | How do adjusted conversion, within-visit choice and considerations per model compare, and how uncertain are they? | The supplied `03_models.html` describes a proposed notebook: conversion LPMs with Logit checks and a Gelbach decomposition; conditional Logit among single-purchase visits; model-level Poisson counts; an optional within-model price extension. Inputs are the saved `01` datasets; outputs are Tables 5.1–5.6, Figure 5.1 and an exhibit index under `results/models/`. The HTML is a review export, not an executable notebook already in this repository. |

The logical sequence is **scope → measurement → description → modelling**. The
technical build starts from the raw behaviour file: `01` needs no output from `00`.
The substantive work is consequent without introducing an unnecessary intermediate
dataset. `00` does not import or execute the other scripts and never requires `01`
or `02` to have been run.

In [1]:
# 01. Load the original sources for scope screening only.
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.max_colwidth', None)

DATA_DIR = Path('..') / 'data' / '01_LG_Electronics_Appliances'
required = ['electronics_market_behavior.csv', 'appliance_brand_reviews.csv', 'DATA_DICTIONARY.md']
missing = [name for name in required if not (DATA_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f'Run from Marketing-Project and check {DATA_DIR}: {missing}')

behavior = pd.read_csv(DATA_DIR / required[0])
reviews = pd.read_csv(DATA_DIR / required[1])


## 3. What the available sources can answer

The local `DATA_DICTIONARY.md` is the source for the following data meanings and
sampling statements; the next cells check coverage in the actual CSVs.

| Source | Observation and useful information | Questions it can support | Boundary |
| --- | --- | --- | --- |
| Behaviour | A product event with user, session, product, brand, category, time and listed price | Observed consideration and purchase journeys; relative reach within the retained brands; association of browsing and conversion | Historical retailer proxy; no product specifications, stock, traffic source, advertising exposure, costs or intervention assignment |
| Reviews | A review with rating, text, date and verification; repeated product listing fields | Satisfaction and review-topic comparisons within represented appliance categories | Self-selected reviewers, different dates and product mix; listing fields are not measured at purchase |

The instructor retained LG and five competitors in each behaviour category and
sampled whole sessions within the curated scope. This does not provide full user
histories, all retailer traffic or population brand shares. Review counts are capped
within brand/category cells and are not a popularity measure. Neither file is LG's
proprietary customer database. The retailer, country and behaviour-price currency
are unidentified; the review listing price is described as USD.

In [2]:
# 02. Check source compatibility without creating an analytical cohort.
required_behavior = {'event_type', 'user_id', 'session_id', 'product_id', 'brand', 'product_category', 'price', 'event_time'}
required_reviews = {'review_id', 'product_id', 'brand', 'product_category', 'review_rating', 'review_text', 'review_date', 'verified_purchase'}
assert required_behavior.issubset(behavior.columns)
assert required_reviews.issubset(reviews.columns)

compatibility = pd.Series({
    'Behaviour categories': ', '.join(sorted(behavior['product_category'].dropna().unique())),
    'Review categories': ', '.join(sorted(reviews['product_category'].dropna().unique())),
    'Shared product-ID labels': len(set(behavior['product_id'].dropna()) & set(reviews['product_id'].dropna())),
    'User ID available in reviews': 'user_id' in reviews.columns,
}, name='Source compatibility')
display(compatibility.to_frame())


,Source compatibility
Behaviour categories,"microwave, monitor, refrigerator, tv, vacuum_cleaner, washing_machine"
Review categories,"air_conditioner, cooking, dishwasher, laundry, other_appliance, refrigerator_freezer"
Shared product-ID labels,0
User ID available in reviews,False


The dictionary explicitly states that the products and users differ and that the
files have no join relationship. An accidental matching ID would not establish a
valid link. Reviews have no TV category, and similarly named appliance categories
are not a product crosswalk. Brand-level averages from reviews cannot be attached
to individual behaviour rows to explain their purchases.

**Decision:** use behaviour for this RQ. Reviews remain a credible *alternative
project*, not an extra explanatory variable for TV conversion. They are not rejected
merely because one column has many missing values.

## 4. Catalogue coverage before selecting a category or competitor

The original exploration compared all retained categories and competitors before
retaining TV/LG–Samsung. The next matrix preserves that broad coverage check. It
counts distinct listed products in the original event file, including products with
no view; it is **not** the official viewed-consideration cohort or a conversion table.
The original CSV is left unchanged. Invalid mappings stop this check rather than
silently introducing a second cleaning policy.

In [3]:
# 03. Inspect the sampled catalogue across all brand-category combinations.
catalog_fields = ['product_id', 'brand', 'product_category']
assert behavior[catalog_fields].notna().all().all()
mapping = behavior.groupby('product_id')[['brand', 'product_category']].nunique()
assert mapping.eq(1).all().all(), 'Resolve product mappings in the official build before interpreting coverage.'

catalogue = behavior[catalog_fields].drop_duplicates()
coverage = pd.crosstab(catalogue['brand'], catalogue['product_category'])
display(coverage)
print('Distinct sampled brands:', len(coverage))
print('Observed LG/competitor category pairs:', int(coverage.drop(index='lg').gt(0).sum().sum()))
print('Categories containing both LG and Samsung:',
      ', '.join(coverage.columns[coverage.loc[['lg', 'samsung']].gt(0).all(axis=0)]))


Distinct sampled brands: 21
Observed LG/competitor category pairs: 30
Categories containing both LG and Samsung: microwave, monitor, refrigerator, tv, vacuum_cleaner, washing_machine


product_category,microwave,monitor,refrigerator,tv,vacuum_cleaner,washing_machine
brand,,,,,,
acer,0,100,0,0,0,0
aoc,0,54,0,0,0,0
arg,4,0,0,0,0,0
artel,0,0,0,38,0,29
atlant,0,0,109,0,0,0
beko,0,0,30,0,0,23
bosch,0,0,0,0,54,0
dauscher,10,0,0,0,0,0
elenberg,5,0,0,0,0,0


A zero denotes missing sample coverage, not absence from the market. Product IDs
identify catalogue items, not detailed specifications. More listed products do not
mean broader effective exposure, stronger demand or greater purchase conversion.

### 4.1 Why TVs, with washing machines as a serious alternative

The scope decision balances the following criteria; none is an automatic winner rule.

| Criterion | What the exploration established or needs checked | Where the exact current evidence belongs |
| --- | --- | --- |
| Sufficient support for both brands | TV and washing-machine comparisons are feasible; washing machines can have more LG purchase outcomes even with fewer combined considerations | `02`, Table 4.1, using the current `01` timing convention |
| Comparable commercial positioning | Price overlap and sampled assortment are useful screens; similar prices do not establish equivalent specifications | Historical first-view-price screening in Evidence E3; consideration-level TV distributions only in `02`, Table 4.2 and Figure 4.2 |
| Robustness to assortment | Considerations can be concentrated in a few products; a large row count can conceal limited independent product support | `01`, top-model decision tables; `02`, Tables 4.7c–f |
| A focused managerial question | One category allows the same kind of observed journey to be studied without pooling unrelated purchase processes | This decision record; the outcomes are measured in `01` |

**Decision:** retain TVs because the exploration jointly supports a substantial
two-brand comparison, a feasible price benchmark and a focused journey question.
Washing machines are a defensible alternative, not a failed hypothesis. We choose
one coherent scope rather than claim that TVs are objectively superior in every
criterion. The exploration occurred before the current pipeline but after looking
at the available data; this is an exploratory choice and must be disclosed.

We do not choose TVs because they have the largest significant gap, the most events
alone, or supposedly the highest LG price. Nor do we pool categories to manufacture
support. Differences in outcome counts, prices and concentration are trade-offs;
no predeclared scoring system existed and none is invented retrospectively.

### 4.2 Why Samsung rather than the other sampled TV brands

The historical all-competitor first-view-price screen is reported in Evidence E3,
with all 30 candidate comparisons embedded. It weights considerations, matching
the original scope exploration. Samsung's TV price proximity, support and assortment
jointly made it a feasible primary comparator; price similarity does not establish
equivalent specifications or shoppers. Current price distributions belong to `02`.

The coverage and historical first-view-price screen support Samsung as a feasible
benchmark. They do not establish that it is the closest competitor for every
individual TV, or require it to rank first in every other category.

Samsung is therefore a feasible primary benchmark within this sample. This does
not establish matched products, identical customer intent or market leadership.
The other TV brands remain in the `01` dataset as contextual alternatives and prior
browsing information; they are not discarded from observed journeys. The group
`other_four` is a convenience, not evidence that its brands share a positioning.

An alternative is to compare LG with all remaining brands pooled.
That benchmark changes when the competitors' price and product mix changes, so it
is useful as context rather than a substitute for a clearly named comparator.

## 5. Review-based alternatives: feasible, but a different project

This section owns the review feasibility exploration, which is
not performed in the TV build or preliminary analysis beyond noting the absence
of TVs. It examines whether a satisfaction question was a real alternative.

In [4]:
# 04. Check review quality and retain ratings/text despite missing listing prices.
review_quality = pd.DataFrame({
    'dtype': reviews.dtypes.astype(str),
    'missing_rows': reviews.isna().sum(),
    'missing_pct': 100 * reviews.isna().mean(),
    'distinct_values': reviews.nunique(),
})
display(review_quality.round(2))
review_dates = pd.to_datetime(reviews['review_date'], errors='coerce')
ratings = pd.to_numeric(reviews['review_rating'], errors='coerce')
prices = pd.to_numeric(reviews['price'], errors='coerce')
text_available = reviews['review_text'].fillna('').astype(str).str.strip().ne('')
review_checks = pd.Series({
    'Exact duplicate review rows': int(reviews.duplicated().sum()),
    'Duplicate review IDs': int(reviews['review_id'].duplicated().sum()),
    'Unparseable or missing review dates': int(review_dates.isna().sum()),
    'Ratings outside the five-star scale or missing': int((~ratings.isin([1, 2, 3, 4, 5])).sum()),
    'Missing or blank review text': int((~text_available).sum()),
    'Usable positive listing prices (%)': 100 * (np.isfinite(prices) & prices.gt(0)).mean(),
})
display(review_checks.to_frame('Value').round(2))
assert not reviews['review_id'].duplicated().any()
assert review_dates.notna().all() and ratings.isin([1, 2, 3, 4, 5]).all()
assert reviews[['product_id', 'brand', 'product_category']].notna().all().all()
assert reviews.groupby('product_id')[['brand', 'product_category']].nunique().eq(1).all().all()


,dtype,missing_rows,missing_pct,distinct_values
review_id,str,0,0.00,20004
product_id,str,0,0.00,2750
brand,str,0,0.00,8
product_category,str,0,0.00,6
product_title,str,0,0.00,2637
price,float64,15883,79.40,141
product_average_rating,float64,0,0.00,40
product_rating_count,int64,0,0.00,167
review_rating,float64,0,0.00,5
review_title,str,2,0.01,16608


,Value
Exact duplicate review rows,0.0
Duplicate review IDs,0.0
Unparseable or missing review dates,0.0
Ratings outside the five-star scale or missing,0.0
Missing or blank review text,21.0
Usable positive listing prices (%),20.6


Missing price prevents some price-based analyses; it does not invalidate an observed
rating or usable text. No price is imputed and no review is removed because its
listing price is unavailable. `review_rating` is a row-level response;
`product_average_rating` and `product_rating_count` are repeated listing-level
information and must not be counted as independent reviewer outcomes.

In [5]:
# 05. Map category support and the observation window for a satisfaction study.
review_work = reviews.assign(
    brand=reviews['brand'].astype('string').str.strip().str.lower(),
    observed_date=review_dates,
    rating=ratings,
    usable_price=np.isfinite(prices) & prices.gt(0),
    usable_text=text_available,
)
review_coverage = review_work.groupby(['brand', 'product_category']).agg(
    reviews=('review_id', 'size'), products=('product_id', 'nunique'),
).unstack('product_category', fill_value=0)
display(review_coverage)
review_support = review_work.groupby('product_category').agg(
    reviews=('review_id', 'size'), products=('product_id', 'nunique'),
    brands=('brand', 'nunique'), first_review=('observed_date', 'min'),
    last_review=('observed_date', 'max'), usable_price_share=('usable_price', 'mean'),
    usable_text_share=('usable_text', 'mean'),
)
review_support[['usable_price_share', 'usable_text_share']] *= 100
display(review_support.rename(columns={
    'usable_price_share': 'usable_price_pct', 'usable_text_share': 'usable_text_pct',
}).round({'usable_price_pct': 2, 'usable_text_pct': 2}))


reviews                                             \
product_category air_conditioner cooking dishwasher laundry other_appliance   
brand                                                                         
bosch                          0     205        611     204               0   
electrolux                     0     153         63     230               8   
frigidaire                     7    1287        446     263            1287   
ge                            16    1287        330     616            1287   
haier                         10      14          0    1287             130   
lg                             0     327         88    1287               0   
samsung                        0     568        210     613               6   
whirlpool                      0     272        211     806              41   

                                             products                     \
product_category refrigerator_freezer air_conditioner cooking dishwasher   
brand                                                                      
bosch                             150               0      71         72   
electrolux                        112               0      42          7   
frigidaire                       1287               1     210         81   
ge                                939               2     279         87   
haier                             422               2       6          0   
lg                               1221               0      57         13   
samsung                          1287               0      81         26   
whirlpool                         416               0      81         56   

                                                               
product_category laundry other_appliance refrigerator_freezer  
brand                                                          
bosch                 31               0                   16  
electrolux            53               3                   45  
frigidaire            45              19                  112  
ge                   111              12                  195  
haier                 21              12                   32  
lg                   186               0                  165  
samsung              138               2                  166  
whirlpool            136               7                   69

,reviews,products,brands,first_review,last_review,usable_price_pct,usable_text_pct
product_category,,,,,,,
air_conditioner,33,5,3,2017-09-26,2022-05-30,21.21,100.00
cooking,4113,827,8,2006-08-30,2023-08-08,15.78,99.85
dishwasher,1959,342,7,2006-07-30,2023-07-24,4.24,100.00
laundry,5306,721,8,2005-04-15,2023-08-23,4.05,99.91
other_appliance,2759,55,6,2004-07-12,2023-09-01,57.05,99.78
refrigerator_freezer,5834,800,8,2005-09-12,2023-08-18,27.31,99.93


In [6]:
# 06. Describe ratings and verification without treating either as product quality.
display(pd.crosstab(review_work['brand'], review_work['rating']).rename_axis(columns='Review stars'))
verified_text = reviews['verified_purchase'].astype(str).str.strip().str.lower()
assert verified_text.isin(['true', 'false', '1', '0']).all(), 'Check the source verification encoding.'
review_work['verified'] = verified_text.isin(['true', '1'])
display(pd.crosstab(review_work['brand'], review_work['verified']).rename(
    columns={False: 'Unverified reviews', True: 'Verified reviews'}))

review_candidates = review_work[review_work['brand'].isin(['lg', 'samsung'])].groupby(
    ['product_category', 'brand']
).agg(reviews=('review_id', 'size'), products=('product_id', 'nunique'),
      mean_rating=('rating', 'mean'), verified_share=('verified', 'mean'),
      first_review=('observed_date', 'min'), last_review=('observed_date', 'max'))
review_candidates['verified_share'] *= 100
assert set(review_candidates.index.get_level_values('brand')) == {'lg', 'samsung'}
display(review_candidates.rename(columns={'verified_share': 'verified_pct'}).round(
    {'mean_rating': 2, 'verified_pct': 2}))


Review stars,1.0,2.0,3.0,4.0,5.0
brand,,,,,
bosch,527,131,73,111,328
electrolux,304,63,23,46,130
frigidaire,1253,290,335,497,2202
ge,1658,384,269,434,1730
haier,258,87,127,285,1106
lg,1229,216,170,282,1026
samsung,1601,267,136,194,486
whirlpool,968,141,87,160,390


verified,Unverified reviews,Verified reviews
brand,,
bosch,887,283
electrolux,411,155
frigidaire,806,3771
ge,1904,2571
haier,490,1373
lg,1901,1022
samsung,2270,414
whirlpool,1307,439


reviews  products  mean_rating  verified_pct  \
product_category     brand                                                   
cooking              lg           327        57         3.31         54.13   
                     samsung      568        81         2.46         23.77   
dishwasher           lg            88        13         1.64          1.14   
                     samsung      210        26         1.50          4.76   
laundry              lg          1287       186         2.91         27.89   
                     samsung      613       138         2.07         16.15   
other_appliance      samsung        6         2         2.33         66.67   
refrigerator_freezer lg          1221       165         2.84         39.72   
                     samsung     1287       166         2.14         12.90   

                             first_review last_review  
product_category     brand                             
cooking              lg        2006-12-12  2023-05-21  
                     samsung   2009-09-09  2023-07-25  
dishwasher           lg        2006-07-30  2023-07-12  
                     samsung   2010-07-01  2023-07-08  
laundry              lg        2006-11-09  2023-08-12  
                     samsung   2009-08-07  2023-07-10  
other_appliance      samsung   2022-04-28  2023-05-23  
refrigerator_freezer lg        2008-03-03  2023-07-07  
                     samsung   2008-08-19  2023-06-26

The rating and verification tables reproduce the *questions raised* by the old
review exploration: do raw brand patterns persist within the same categories,
periods, products and verification groups? Unequal extremes in the rating
distribution are not population defect rates. Unverified does not mean fake, and
verified does not prove an unbiased review. Long time coverage is not a balanced
panel; dates, sampled products and review selection differ.

A separate satisfaction project could compare ratings within overlapping appliance
categories and then inspect text themes, with product dependence and verified-only
checks. It would need its own cleaning/measurement decisions and an appropriate
course-method justification. These descriptive tables do not establish adjusted
brand quality differences or explain the retailer's TV purchase gap.

## 6. Research directions considered and reasons for the chosen route

| Direction considered | Real merit | Why it is not the primary question here | What survives |
| --- | --- | --- | --- |
| Two stage gaps: view→cart and cart→purchase | Localises observed progression; motivated the first models in `Edoardo_notebook` | Different denominators; cart entry selects shoppers; omits the reach component of the purchase-count shortfall | Diagnostic funnel in `02`; conditional stage models only if needed later |
| Purchase conversion adjusted only for price and month | One binary commercial endpoint; simple course Logit | Leaves observed browsing composition and the count/reach distinction unexamined | A baseline specification, not the final explanation |
| Purchase gap after prior cross-brand browsing | Uses information available before the consideration; follows the shopper-mix exploration | A changed adjusted estimate does not establish a cause or a profitable targeting action | Supporting question and a timed browsing specification |
| Whole-period single-brand versus cross-brand shoppers | Exposes that brands can attract different observed browsing profiles | Uses later behaviour and incomplete histories; group membership is selected | Descriptive lesson and optional retrospective sensitivity, not a first-view predictor |
| Washing-machine conversion | Credible sample and competitor alternative | A different scope, without a demonstrated reason to replace the retained TV investigation | Explicit alternative in the scope rationale |
| Other behaviour categories or all competitors pooled | Tests whether the TV/Samsung comparison was arbitrary | Support, price comparability, assortment and purchase processes vary | All-category/competitor screening; category comparisons in `02` |
| Appliance satisfaction and review topics | Usable ratings/text in overlapping LG/Samsung appliance categories | Different products, users, periods and managerial outcome; no TV linkage | Review feasibility evidence above |
| Segmentation/targeting, RFM or K-means | Could organise observed browsing if a targeting decision required it | Adds a new objective; no demographics or complete customer histories; clusters do not explain causes | Simple observed browsing variables instead of a new primary project |
| Pricing elasticity, optimal discounts or ROI | Direct economic decision | No exogenous price variation, complete demand exposure, costs or margins; listed prices also vary with product mix | Initial price as an associational adjustment, not an optimisation tool |

**Retained route:** explain the observed purchase-count shortfall with an accounting
breakdown, inspect where the descriptive differences sit, then assess the conversion
comparison with a transparent purchase model. The alternatives are excluded for
fit to this decision and data, not because their estimated gaps or p-values were
less attractive. This does not certify TV as the only defensible research project.

## Evidence record: what the exploratory work actually found

This record is readable without opening the retired notebooks or their code. Each
block states the question, the observed result, its limit and the decision it informed.
The historical analyses were **re-executed on 18 November 2026 against the original
local CSVs** before these summaries were generated. Numerical values below are
**fixed historical snapshots**, not results recomputed by the current screening cells.
The archived aggregate tables are embedded here, so the original files are optional
implementation provenance, not reading or execution dependencies.

The historical consideration definition required cart/purchase events **strictly
later** than first view. `01` now accepts same-second carts/purchases. Raw event
counts also differ before/after deduplication. Whole-period shopper types and
user-only uncertainty in the old experiments differ from the current pipeline.
Therefore these numbers explain **how the project developed**, not what the final
report must reproduce. Current descriptive results belong to `02`, models to `03`.
The compact summaries are the main reading path; expandable tables retain the
supporting aggregate results without requiring anyone to inspect Python.

### E1. Were both sources usable, and could they be combined?

**Question and method.** Inventory schemas, missingness, dates, duplicate rows,
identifier mappings and category coverage before choosing an outcome.

**Result.** Behaviour contained 649,605 events, eight fields, 21 brands and 1,613
products over October–November 2019. Removing 1,247 exact copies left 648,358 events;
no invalid rows were quarantined under the exploratory rules. Four session IDs
were shared across users, while product-to-brand/category mappings were consistent.
Reviews contained 20,004 rows, 14 fields and 2,750 products across eight brands,
from July 2004 to September 2023; no duplicates/invalid reviews were removed.
Listing price was missing in 79.4%, while ratings and verification were complete.
There were six missing text values; the more explicit current blank-string check
also finds whitespace-only content, giving 21 missing-or-blank texts.

**Conclusion and decision.** Use the composite user/session key and an audited
build, rather than count events as independent purchases or people. Reviews have
no TV category and no valid product/user join to behaviour; use them only for a
separate satisfaction question. Missing listing prices do not make ratings/text
unusable. The official cleaning and reconciliation are implemented only in `01`.

**Limit.** These checks establish consistency under stated rules, not complete
histories, representative market coverage, absence of bots or truthful reviews.

### E2. Which categories could sustain the project?

**Question and method.** Compare all six categories using observed activity, sample
support, prices and concentration; then compare LG with Samsung within each category.

**Result.** TVs had the most cleaned events (211,958), followed by washing machines
(161,871), vacuum cleaners (117,983), refrigerators (108,259), microwaves (26,569)
and monitors (21,718). Every category contained six retained brands. November had
more events in every category, motivating time adjustment without identifying
promotions or a seasonal cause. The historical all-category build yielded 363,277
viewed considerations.

TVs supplied the largest combined LG/Samsung consideration and purchase sample:
74,712 rows, 196 products and 2,150 purchased considerations. Washing machines were
a serious alternative: LG had 544 purchases versus 505 in TVs, although the combined
sample was smaller. Monitors offered close prices but only 13 observed LG purchase
outcomes. LG microwave activity was concentrated in its five leading models (89.4%);
LG refrigerator and vacuum-cleaner concentration was about 60%, versus 42.3% in TVs.
The washing-machine top-five share was 51.2%, and monitor share 31.9%.

**Conclusion and decision.** Retain TVs for the joint support/comparator/focus case;
do not claim they maximise every criterion. Washing machines were a viable route,
monitors had limited purchase support, and other categories brought different
support, concentration and price trade-offs. LG refrigerators had a higher median
price than LG TVs, so 'TVs are LG's most expensive category' was not a justification.
The all-category gap results were exploratory, not a significance contest.

**Limit.** A category's event volume is neither conversion nor independent sample
size. Product specifications and availability remain unobserved.

<details>
<summary>Historical category-by-brand support, prices and concentration</summary>

| Category | Brand | Considerations | Purchases | Products | Median first-view price | Top-five share (%) |
| --- | --- | --- | --- | --- | --- | --- |
| microwave | arg | 3,427 | 160 | 4 | 59.180 | 100.000 |
| microwave | dauscher | 1,773 | 33 | 10 | 58.950 | 92.047 |
| microwave | elenberg | 1,320 | 93 | 5 | 51.460 | 100.000 |
| microwave | lg | 2,822 | 71 | 17 | 94.960 | 89.405 |
| microwave | midea | 1,523 | 44 | 11 | 48.880 | 87.656 |
| microwave | samsung | 4,701 | 150 | 29 | 125.870 | 62.689 |
| monitor | acer | 4,614 | 66 | 100 | 180.045 | 31.773 |
| monitor | aoc | 1,099 | 6 | 54 | 230.920 | 33.940 |
| monitor | hp | 1,597 | 24 | 82 | 218.540 | 50.407 |
| monitor | lg | 1,312 | 13 | 40 | 221.320 | 31.860 |
| monitor | qmax | 560 | 9 | 28 | 84.810 | 47.857 |
| monitor | samsung | 3,718 | 59 | 61 | 223.630 | 48.521 |
| refrigerator | atlant | 9,219 | 142 | 109 | 257.380 | 33.008 |
| refrigerator | beko | 9,794 | 210 | 30 | 243.350 | 66.765 |
| refrigerator | indesit | 17,715 | 363 | 70 | 302.430 | 36.912 |
| refrigerator | lg | 13,079 | 191 | 49 | 535.120 | 59.768 |
| refrigerator | midea | 5,863 | 88 | 88 | 221.340 | 29.012 |
| refrigerator | samsung | 9,950 | 132 | 41 | 643.260 | 49.879 |
| tv | artel | 19,872 | 624 | 38 | 154.190 | 83.887 |
| tv | haier | 9,654 | 340 | 20 | 231.640 | 72.250 |
| tv | kivi | 6,465 | 222 | 19 | 192.930 | 73.782 |
| tv | lg | 22,213 | 505 | 105 | 460.840 | 42.309 |
| tv | samsung | 52,499 | 1,645 | 91 | 493.960 | 52.382 |
| tv | tcl | 5,008 | 156 | 17 | 251.950 | 70.747 |
| vacuum_cleaner | bosch | 10,649 | 306 | 54 | 115.810 | 56.512 |
| vacuum_cleaner | lg | 5,493 | 132 | 27 | 172.380 | 60.004 |
| vacuum_cleaner | samsung | 26,308 | 906 | 73 | 92.640 | 50.437 |
| vacuum_cleaner | tefal | 6,170 | 133 | 49 | 138.970 | 66.499 |
| vacuum_cleaner | vitek | 5,628 | 137 | 26 | 77.170 | 66.827 |
| vacuum_cleaner | xiaomi | 8,261 | 139 | 24 | 401.300 | 75.935 |
| washing_machine | artel | 4,116 | 120 | 29 | 93.750 | 46.064 |
| washing_machine | beko | 8,967 | 285 | 23 | 212.850 | 82.826 |
| washing_machine | indesit | 16,803 | 412 | 45 | 194.900 | 50.199 |
| washing_machine | lg | 21,959 | 544 | 69 | 380.960 | 51.164 |
| washing_machine | midea | 6,442 | 265 | 28 | 120.720 | 79.246 |
| washing_machine | samsung | 32,684 | 910 | 48 | 347.470 | 58.405 |

</details>

### E3. Why Samsung rather than the other competitors?

**Question and method.** Screen all 30 observed LG/competitor category pairs by
support, median first-view-price distance, central-price-range overlap and concentration.
The historical screen weighted considerations, not products equally.

**Result.** Samsung was closest in median price in five categories; Tefal was closer
for vacuum cleaners. Within TVs, Samsung had the closest median to LG (493.96
versus 460.84, a 7.19% distance), the only positive overlap of the central price
ranges, and the broadest competitor assortment (91 products). LG had 105 products;
Samsung supplied 52,499 considerations versus LG's 22,213. Artel, Haier, Kivi and
TCL were observed alternatives but less close in this price-positioning screen.

**Conclusion and decision.** Select Samsung as a named, feasible comparator and
retain the other brands as context and browsing history. Pooling all rivals changes
the price/product composition of the benchmark and is less specific managerially.
Category choice and competitor choice need separate arguments.

**Limit.** Neither price proximity nor central-range overlap matches specifications,
stock or shopper intent. The newer equal-product catalogue screen was a different
summary; it cannot replace this historical result or its weighting. Current TV
first-view-price distributions belong to `02`.
<details>
<summary>All 30 historical competitor screens (LG/rival support is shown above)</summary>

| Category | Rival | LG median | Rival median | Distance (%) | Central overlap width | Combined considerations | Combined purchases |
| --- | --- | --- | --- | --- | --- | --- | --- |
| microwave | samsung | 94.960 | 125.870 | 32.551 | 0.000 | 7,523 | 221 |
| microwave | arg | 94.960 | 59.180 | 37.679 | 0.000 | 6,249 | 231 |
| microwave | dauscher | 94.960 | 58.950 | 37.921 | 0.000 | 4,595 | 104 |
| microwave | elenberg | 94.960 | 51.460 | 45.809 | 0.000 | 4,142 | 164 |
| microwave | midea | 94.960 | 48.880 | 48.526 | 0.000 | 4,345 | 115 |
| monitor | samsung | 221.320 | 223.630 | 1.044 | 217.912 | 5,030 | 72 |
| monitor | hp | 221.320 | 218.540 | 1.256 | 160.110 | 2,909 | 37 |
| monitor | aoc | 221.320 | 230.920 | 4.338 | 187.020 | 2,411 | 19 |
| monitor | acer | 221.320 | 180.045 | 18.649 | 139.020 | 5,926 | 79 |
| monitor | qmax | 221.320 | 84.810 | 61.680 | 17.977 | 1,872 | 22 |
| refrigerator | samsung | 535.120 | 643.260 | 20.209 | 232.440 | 23,029 | 323 |
| refrigerator | indesit | 535.120 | 302.430 | 43.484 | 0.000 | 30,794 | 554 |
| refrigerator | atlant | 535.120 | 257.380 | 51.902 | 0.000 | 22,298 | 333 |
| refrigerator | beko | 535.120 | 243.350 | 54.524 | 0.000 | 22,873 | 401 |
| refrigerator | midea | 535.120 | 221.340 | 58.637 | 0.000 | 18,942 | 279 |
| tv | samsung | 460.840 | 493.960 | 7.187 | 244.390 | 74,712 | 2,150 |
| tv | tcl | 460.840 | 251.950 | 45.328 | 0.000 | 27,221 | 661 |
| tv | haier | 460.840 | 231.640 | 49.735 | 0.000 | 31,867 | 845 |
| tv | kivi | 460.840 | 192.930 | 58.135 | 0.000 | 28,678 | 727 |
| tv | artel | 460.840 | 154.190 | 66.542 | 0.000 | 42,085 | 1,129 |
| vacuum_cleaner | tefal | 172.380 | 138.970 | 19.382 | 64.840 | 11,663 | 265 |
| vacuum_cleaner | bosch | 172.380 | 115.810 | 32.817 | 80.280 | 16,142 | 438 |
| vacuum_cleaner | samsung | 172.380 | 92.640 | 46.258 | 28.080 | 31,801 | 1,038 |
| vacuum_cleaner | vitek | 172.380 | 77.170 | 55.233 | 4.600 | 11,121 | 269 |
| vacuum_cleaner | xiaomi | 172.380 | 401.300 | 132.800 | 0.000 | 13,754 | 271 |
| washing_machine | samsung | 380.960 | 347.470 | 8.791 | 86.660 | 54,643 | 1,454 |
| washing_machine | beko | 380.960 | 212.850 | 44.128 | 0.000 | 30,926 | 829 |
| washing_machine | indesit | 380.960 | 194.900 | 48.840 | 0.000 | 38,762 | 956 |
| washing_machine | midea | 380.960 | 120.720 | 68.312 | 0.000 | 28,401 | 809 |
| washing_machine | artel | 380.960 | 93.750 | 75.391 | 0.000 | 26,075 | 664 |

</details>

### E4. Was satisfaction a genuine alternative?

**Question and method.** Inspect review support by brand/category/date, star ratings,
verification and price/text availability rather than reject the file wholesale.

**Result.** Refrigerators/freezers and laundry contained 5,834 and 5,306 reviews;
cooking had 4,113 reviews and the largest product variety (827 products), dishwasher
1,959, other appliances 2,759, and air conditioners only 33 reviews across five
products. LG and Samsung overlapped in cooking, dishwashers, laundry and
refrigerators/freezers. Samsung had 1,601 one-star versus 486 five-star reviews;
LG had 1,229 versus 1,026. Verified reviews numbered 414 of 2,684 for Samsung and
1,022 of 2,923 for LG. Both therefore had more unverified than verified reviews.

**Conclusion and decision.** A satisfaction/text project was feasible in overlapping
appliance categories, with category/time/product adjustment and a verified-only
sensitivity. It would answer a different managerial question, so it was not joined
to the TV conversion study. The review tables in Section 5 retain the full rating,
verification, coverage and candidate summaries independently of the retired file.

**Limit.** Review caps and self-selection prevent market-share or defect-rate claims.
Unverified is not synonymous with fake. Raw brand ratings do not establish an
adjusted quality difference or explain individual TV purchases.

### E5. What did the initial journey benchmark reveal?

**Question and method.** Start from viewed user/session/product considerations and
compare observed view→cart, eligible-cart→purchase and view→purchase rates, then
check month, price overlap and repetition.

**Result.** LG's historical rates were 4.83%, 34.86% and 2.27%, against Samsung's
6.31%, 40.17% and 3.13%. The gaps were -1.48, -5.31 and -0.86 percentage points.
There were 446 purchased considerations without a qualifying cart path, so multiplying
the two stage rates did not recover overall purchase conversion. There were 1,114
first views on the last observed day, motivating an observation-window check.

The stage gaps appeared in both months but changed size: LG's view→cart rate rose
from 3.10% to 5.59%, Samsung's from 5.69% to 6.64%; cart→purchase fell from 44.76%
to 32.44% for LG and from 46.25% to 37.33% for Samsung. Central prices overlapped
substantially. Repetition mattered: 15,561 users contributed multiple rows, accounting
for 72.82% of considerations. Five products generated 42.31% of LG rows and 52.38%
of Samsung rows.

**Conclusion and decision.** Price/time adjustment, explicit stage denominators,
observation-window checks and dependence-aware uncertainty were needed before
interpreting a brand gap. The funnel became a diagnostic; the recent primary RQ
places it inside the broader purchased-count shortfall rather than equating conversion
with the whole business problem.

**Limit.** A zero is no qualifying observed transition in that unit; it is not proof
of no eventual purchase. The denominator changes after cart entry; a cart-stage
gap does not identify checkout friction. `02` owns current funnel/period exhibits.
<details>
<summary>Historical monthly stage results</summary>

| Brand | Month | Considerations | Carts | Purchases after cart | View→cart (%) | Cart→purchase (%) |
| --- | --- | --- | --- | --- | --- | --- |
| lg | 2019-10 | 6,770 | 210 | 94 | 3.102 | 44.762 |
| lg | 2019-11 | 15,443 | 863 | 280 | 5.588 | 32.445 |
| samsung | 2019-10 | 18,494 | 1,053 | 487 | 5.694 | 46.249 |
| samsung | 2019-11 | 34,005 | 2,258 | 843 | 6.640 | 37.334 |

</details>

### E6. Did popular products or the shopper selection change the picture?

**Question and method.** Remove each brand's single most-considered product, and
separately restrict to users who considered both brands during the two months.

**Result.** Removing LG P000640 (2,723 considerations) and Samsung P000507 (9,106)
left 19,490 and 43,393 rows. View→cart became 4.64% versus 5.61%, and cart→purchase
34.81% versus 38.83%: the gaps narrowed, but the assortment also changed.
The whole-period dual-brand group contained 6,364 users and 28,027 rows. Its
view→purchase rates were 1.386% for LG and 1.421% for Samsung, a gap of about
-0.035 pp. View→cart differed by -0.042 pp; cart→purchase was +0.859 pp for LG.
The dual-brand category comparisons are preserved in E10's supporting results.

**Conclusion and decision.** Aggregate comparisons were sensitive to assortment
and selected audiences. Investigate concentration and observed shopper mix rather
than attribute the raw gap directly to pages or checkout. The full sample remained
the reference; restricted samples were sensitivities.

**Limit.** These are the same observed users in both brand denominators, not matched
products/times/intent or a within-person experiment. Whole-period membership uses
future information relative to early views. Near-zero estimates do not establish
equivalence. Product exclusions answer a narrower question.

### E7. What were the shopper types, and why did mix matter?

**Question and method.** Label users LG-only, Samsung-only or both-brand over the
full observation window; compare group rates and descriptive user profiles.

**Result.** There were 6,544 LG-only users, 22,963 Samsung-only users and 6,364
cross-shoppers. Cross-shoppers contributed 54.90% of LG considerations but 30.16%
of Samsung considerations. Their per-consideration purchase rates were about
1.39%/1.42%, below the single-brand rates of 3.35% for LG-only and 3.87% for Samsung-only.
Cross-shoppers considered a mean of 4.40 TVs, versus 1.53/1.60 in the single-brand
groups; they also covered more products, brands and sessions. Their probability of
any recorded TV purchase was 6.32%, versus 5.41%/6.41%: lower conversion per row
was not equivalent to a uniquely low purchase propensity per user.

**Conclusion and decision.** Row-denominator dilution and mix were credible reasons
to examine browsing state before interpreting an average conversion difference.
Use user-level and consideration-level summaries for their respective questions;
do not turn these behavioural labels into demographic or intent segments.

**Limit.** Whole-period profiles are retrospective; missing sampled sessions can
change membership. Different groups do not identify the mechanism behind purchase.
<details>
<summary>Historical group KPIs and complete user-profile summary</summary>

| Group | Brand | Users | Considerations | View→cart (%) | Cart→purchase (%) | View→purchase (%) |
| --- | --- | --- | --- | --- | --- | --- |
| lg_only | lg | 6,544 | 10,019 | 6.318 | 37.441 | 3.354 |
| samsung_only | samsung | 22,963 | 36,666 | 7.454 | 42.261 | 3.873 |
| cross_shopper | lg | 6,364 | 12,194 | 3.608 | 31.136 | 1.386 |
| cross_shopper | samsung | 6,364 | 15,833 | 3.651 | 30.277 | 1.421 |

| shopper_type | users | median_tv_considerations | mean_tv_considerations | mean_tv_sessions | mean_distinct_tv_products | mean_tv_brands_viewed_all_6 | pct_browsed_other_categories | median_price_viewed | pct_users_with_a_purchase_here | pct_users_bought_any_tv |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| lg_only | 6,544 | 1.000 | 1.531 | 1.084 | 1.473 | 1.275 | 11.293 | 450.200 | 4.630 | 5.410 |
| samsung_only | 22,963 | 1.000 | 1.597 | 1.078 | 1.548 | 1.292 | 10.286 | 488.800 | 5.766 | 6.406 |
| cross_shopper | 6,364 | 3.000 | 4.404 | 1.343 | 4.202 | 2.635 | 13.828 | 468.962 | 5.374 | 6.317 |

</details>

### E8. What did reweighting and the shopper-type Logits change?

**Question and method.** Apply Samsung's group weights to LG's group-specific rates;
then compare price/month Logits with a cross-shopper indicator and a brand×group
interaction, separately for three outcomes. For cart→purchase, use cart shares.

**Result.** Direct reweighting changed the purchase gap from -0.860 to -0.373 pp,
a 56.60% accounting contribution under that reference weighting. The corresponding
contributions were 45.40% for view→cart and 27.94% for cart→purchase.
Adding retrospective cross-shopper status to the price/month Logit moved the
standardised purchase gap from -0.827 to -0.281 pp; view→cart moved from -1.563 to
-0.772 pp and cart→purchase from -4.161 to -1.878 pp. With an interaction, purchase
gaps were -0.448 pp among single-brand users and -0.049 pp among cross-shoppers;
the purchase interaction p-value was 0.461. The view→cart interaction p-value was
0.033 and cart→purchase 0.177, under the old user-only covariance convention.

**Conclusion and decision.** The baseline estimate depended materially on how
observed browsing composition was represented. This motivated a prior-only exposure
check, not a declaration that a measured percentage of lost purchases was caused
by audience mix. The newer within-session mix analysis in `02` uses a different
classification; its shares must not be substituted for these whole-period results.

**Limit.** Weights are reference-dependent; the groups include future behaviour.
The model contrasts lacked probability-gap intervals and user-only clustering did
not jointly handle product dependence. Do not infer equivalence from nonsignificance.
<details>
<summary>Historical reweighting and all three model-sequence outcomes</summary>

| kpi | LG_raw_pct | LG_with_Samsung_mix_pct | Samsung_pct | raw_gap_pp | gap_after_mix_pp | share_of_gap_explained_by_mix_pct |
| --- | --- | --- | --- | --- | --- | --- |
| view_to_cart_pct | 4.831 | 5.501 | 6.307 | -1.476 | -0.806 | 45.404 |
| cart_to_purchase_pct | 34.856 | 36.340 | 40.169 | -5.314 | -3.829 | 27.941 |
| view_to_purchase_pct | 2.273 | 2.760 | 3.133 | -0.860 | -0.373 | 56.602 |

| outcome | N | positives | M1_adjusted_gap_pp | M2_adjusted_gap_pp | M3_gap_single_brand_pp | M3_gap_cross_shoppers_pp | M3_interaction_p | all_converged |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| view_to_purchase (Y) | 74,712 | 2,150 | -0.827 | -0.281 | -0.448 | -0.049 | 0.461 | 1 |
| view_to_cart | 74,712 | 4,384 | -1.563 | -0.772 | -1.275 | -0.111 | 0.033 | 1 |
| cart_to_purchase | 4,384 | 1,704 | -4.161 | -1.878 | -3.534 | 1.563 | 0.177 | 1 |

</details>

### E9. Did the lesson survive using only exposure before the first view?

**Question and method.** Replace future cross-shopper membership with whether the
other main brand had already been observed for the user strictly before that row's
first view, including earlier recorded sessions. Fit on the same stage samples.

**Result.** Earlier other-main-brand exposure was recorded for 41.63% of LG rows
versus 14.47% of Samsung rows. The standardised purchase gap moved from -0.827
to -0.486 pp; view→cart from -1.563 to -1.030 pp; cart→purchase from -4.161 to
-3.355 pp. All these fits converged.

**Conclusion and decision.** Browsing history deserved attention even when future
behaviour was excluded, but the attenuation was smaller than with whole-period
labels. The official build therefore prioritises predictors timed before the first
view and separates within-session history from incomplete cross-session history.
This supports including browsing in the analysis; it does not resolve which
managerial intervention would work or make the old stage RQ permanently preferable.

**Limit.** This is not identical to the current `other_brand_opened_before`, which
uses any other TV brand within the same session. Seven elapsed days do not recover
unsampled history; a zero means no earlier exposure recorded, not none experienced.
<details>
<summary>Historical prior-exposure model results</summary>

| outcome | M1_adjusted_gap_pp | M2_prior_exposure_gap_pp | prior_exposure_odds_ratio | converged |
| --- | --- | --- | --- | --- |
| view_to_purchase (Y) | -0.827 | -0.486 | 0.569 | 1 |
| view_to_cart | -1.563 | -1.030 | 0.649 | 1 |
| cart_to_purchase | -4.161 | -3.355 | 0.849 | 1 |

</details>

### E10. What did the broader category and model experiments add?

**Question and method.** Compare LG against Samsung and pooled rivals in every
category, repeat the comparison among dual-brand users, and on TVs add pre-view
browsing counts, prior sessions, a user conditional Logit and price-quintile interactions.

**Result.** The raw purchase gap against Samsung was -0.675 pp for microwaves,
-0.596 for monitors, +0.134 for refrigerators, -0.860 for TVs, -1.041 for vacuum
cleaners and -0.307 for washing machines. Among selected dual-brand users it became
+0.124, +0.166, -0.227, -0.035, +0.113 and +0.261 pp respectively: audience
restriction could change the sign, not just the magnitude. TVs were not the largest
raw-gap category. Against pooled non-LG rivals, LG's refrigerator gap was negative
even though it was positive against Samsung, illustrating comparator sensitivity.

For TVs, the model sequence moved from -0.860 pp (brand only) to -0.827 pp
(price/month) to -0.515 pp with pre-view covariates. LG rows had more products
opened before them (2.023 versus 1.346), more prior brands (1.010 versus 0.728),
more prior sessions (0.402 versus 0.224) and more within-session rival-main-brand
exposure (36.3% versus 11.6%). The last flag differs from E9's across-session
prior-exposure definition. The selected user conditional Logit used 1,697 rows
and gave LG OR 0.944, p=0.618, with its model-based uncertainty. The price-quintile
experiment found lower raw LG conversion in every quintile; individual interaction
p-values ranged from 0.112 to 0.736, without establishing homogeneous responses.
No joint interaction test or causal price interpretation is inferred here. The complete reproduced sequence, conditional
user estimate and quintile results are retained below rather than reduced to a
preferred p-value. Only **332 dual-brand users with both purchase outcomes** supplied
the selected user conditional Logit. Related stage samples had 803 users with
varying cart outcomes and 36 with varying cart→purchase outcomes.

**Conclusion and decision.** Retain the broad screen as a check against arbitrary
scope, take pre-view history seriously, and use fixed effects or price interactions
only to address a stated concern. A conditional user model estimates a selected
within-user association; it does not eliminate brand-specific preferences, and its
nonsignificance cannot establish that LG and Samsung convert equivalently.

Historical p-values printed as 0.000 below mean **p < 0.001**, not exactly zero.
The user conditional Logit uses model-based errors; the other historical Logits
use user-clustered errors.

**Limit.** The original category z-statistics treated rows as independent and are
not valid final significance rankings. The historical price-quintile model was
exploratory and used user-only clustering. No specifications, causal price variation
or complete histories were recovered. These experiments did not determine a final
model or establish an optimal pricing action.
<details>
<summary>Broader historical category comparisons and model experiments</summary>

| category | category_rows | category_median_price | lg_purchases | lg_v2p_pct | samsung_v2p_pct | others_v2p_pct | gap_vs_samsung_pp | gap_vs_others_pp |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| microwave | 15,566 | 84.170 | 71 | 2.516 | 3.191 | 3.766 | -0.675 | -1.251 |
| monitor | 12,900 | 209.910 | 13 | 0.991 | 1.587 | 1.415 | -0.596 | -0.424 |
| refrigerator | 65,620 | 342.070 | 191 | 1.460 | 1.327 | 1.780 | 0.134 | -0.319 |
| tv | 115,711 | 411.050 | 505 | 2.273 | 3.133 | 3.195 | -0.860 | -0.921 |
| vacuum_cleaner | 62,509 | 118.150 | 132 | 2.403 | 3.444 | 2.843 | -1.041 | -0.440 |
| washing_machine | 90,971 | 308.860 | 544 | 2.477 | 2.784 | 2.886 | -0.307 | -0.409 |

**m1**

| Term | Coefficient (log odds) | Historical p-value |
| --- | --- | --- |
| Intercept | -3.431 | 0.000 |
| LG | -0.330 | 0.000 |

**m2**

| Term | Coefficient (log odds) | Historical p-value |
| --- | --- | --- |
| Intercept | -0.522 | 0.102 |
| LG | -0.317 | 0.000 |
| log_price | -0.439 | 0.000 |
| november | -0.277 | 0.000 |

**m3**

| Term | Coefficient (log odds) | Historical p-value |
| --- | --- | --- |
| Intercept | -0.784 | 0.015 |
| LG | -0.193 | 0.002 |
| log_price | -0.362 | 0.000 |
| november | -0.274 | 0.000 |
| pre_products | -0.055 | 0.068 |
| pre_brands | -0.272 | 0.000 |
| other_brand_tv | -0.166 | 0.068 |
| prior_sessions | 0.013 | 0.439 |

**m4**

| Term | Coefficient (log odds) | Historical p-value |
| --- | --- | --- |
| LG | -0.058 | 0.618 |
| log_price | -0.213 | 0.347 |
| november | 0.095 | 0.822 |

**m5**

| Term | Coefficient (log odds) | Historical p-value |
| --- | --- | --- |
| Intercept | -3.086 | 0.000 |
| C(price_quintile)[T.1] | 0.051 | 0.509 |
| C(price_quintile)[T.2] | -0.201 | 0.011 |
| C(price_quintile)[T.3] | -0.110 | 0.143 |
| C(price_quintile)[T.4] | -0.679 | 0.000 |
| LG | -0.443 | 0.000 |
| LG:C(price_quintile)[T.1] | 0.052 | 0.736 |
| LG:C(price_quintile)[T.2] | 0.282 | 0.112 |
| LG:C(price_quintile)[T.3] | 0.192 | 0.261 |
| LG:C(price_quintile)[T.4] | 0.089 | 0.657 |
| november | -0.299 | 0.000 |

| price_quintile | lg | samsung |
| --- | --- | --- |
| 0 | 2.342 | 3.895 |
| 1 | 2.645 | 3.617 |
| 2 | 2.511 | 3.069 |
| 3 | 2.510 | 3.219 |
| 4 | 1.315 | 1.874 |

</details>

<details>
<summary>Additional reproduced aggregate comparisons (no Python source)</summary>

**Stage gaps LG minus Samsung by category (pp)**

```text
view_to_cart  cart_to_purchase  view_to_purchase  lg_carts  samsung_carts
category                                                                                  
microwave              -0.393            -6.026            -0.675       154            275
monitor                -0.945            -5.676            -0.596        37            140
refrigerator            0.234            -1.027             0.134       421            297
tv                     -1.476            -5.314            -0.860      1073           3311
vacuum_cleaner         -1.110            -4.943            -1.041       285           1657
washing_machine        -0.269            -5.358            -0.307      1018           1603
```

**Same shoppers, both brands: view-to-purchase by category**

```text
dual_brand_users  lg_v2p_pct  samsung_v2p_pct  gap_pp
category                                                              
microwave                     649       1.835            1.711   0.124
monitor                       373       0.600            0.434   0.166
refrigerator                 2224       0.657            0.884  -0.227
tv                           6364       1.386            1.421  -0.035
vacuum_cleaner               1664       1.443            1.330   0.113
washing_machine              4737       1.826            1.565   0.261
```

**Pre-exposure covariates, mean by brand**

```text
pre_products  pre_brands  other_brand_tv  prior_sessions
brand                                                            
lg              2.023       1.010           0.363           0.402
samsung         1.346       0.728           0.116           0.224
```

**Model sequence**

```text
M1 brand only                                    LG coef -0.330 (SE 0.059)  adjusted gap -0.860 pp  N=74,712
M2 + log initial price + month                   LG coef -0.317 (SE 0.059)  adjusted gap -0.827 pp  N=74,712
M3 + pre-exposure shopper covariates             LG coef -0.193 (SE 0.061)  adjusted gap -0.515 pp  N=74,712
                Coef.  Std.Err.      z  P>|z|  [0.025  0.975]
Intercept      -0.784     0.321 -2.443  0.015  -1.413  -0.155
LG             -0.193     0.061 -3.163  0.002  -0.313  -0.073
log_price      -0.362     0.052 -6.982  0.000  -0.463  -0.260
november       -0.274     0.048 -5.703  0.000  -0.368  -0.180
pre_products   -0.054     0.030 -1.825  0.068  -0.113   0.004
pre_brands     -0.272     0.058 -4.676  0.000  -0.386  -0.158
other_brand_tv -0.166     0.091 -1.823  0.068  -0.344   0.013
prior_sessions  0.013     0.016  0.774  0.439  -0.019   0.044

M4 user fixed effects (conditional logit), comparison shoppers with varying Y: 332 users, 1697 rows
            coef    SE     p  odds_ratio
LG        -0.058 0.116 0.618       0.944
log_price -0.213 0.226 0.347       0.808
november   0.095 0.421 0.822       1.099

M5 LG x price quintile (quintile 0 = cheapest):
                           Coef.  Std.Err.      z  P>|z|  [0.025  0.975]
LG                        -0.443     0.116 -3.819  0.000  -0.670  -0.216
LG:C(price_quintile)[T.1]  0.052     0.155  0.337  0.736  -0.252   0.356
LG:C(price_quintile)[T.2]  0.282     0.178  1.588  0.112  -0.066   0.631
LG:C(price_quintile)[T.3]  0.192     0.171  1.125  0.261  -0.143   0.527
LG:C(price_quintile)[T.4]  0.088     0.199  0.444  0.657  -0.302   0.479

Raw view-to-purchase by price quintile and brand (%):
brand             lg  samsung
price_quintile               
0              2.340    3.890
1              2.650    3.620
2              2.510    3.070
3              2.510    3.220
4              1.320    1.870
```

</details>

### E11. Which initial stage models were methodological starting points?

**Question and method.** The first stage Logits used initial log price and month,
with month defined at first view for view→cart and at eligible cart for cart→purchase.

**Result.** Standardised view→cart probabilities were 4.774% for LG and 6.338% for
Samsung, a -1.563 pp gap. Cart→purchase probabilities were 35.718% and 39.879%, a
-4.161 pp gap. Both models converged. They established feasibility, not a complete
explanation: adding browsing history subsequently changed both contrasts, as E8–E9
show. There were no confidence intervals for these probability differences.

**Conclusion and decision.** Preserve the binary-response methodology, explicit
stage populations and probability summaries; rebuild substantive models on current
variables with appropriate uncertainty. The supplied `03` now uses LPMs with Logit
checks for conversion, selected-visit choice modelling and a separate count model.
It is not necessary to keep a second official implementation of these old fits.

**Limit.** Coefficient intervals are not probability-gap intervals; user-only
clustering leaves product dependence unresolved. Separate stage probabilities do
not identify causes of abandonment or multiply to all purchases.

### E12. How did the decisions follow from the complete record?

1. **One coherent source:** usable behavioural journeys answered the selected
   retailer question; review coverage supported a different satisfaction project.
2. **A defensible scope:** broad category/comparator screens supported TVs and
   Samsung while acknowledging washing machines and other alternatives.
3. **One measurement contract:** repetitions, shared session IDs, timing and
   concentration required the explicit datasets and audit now implemented by `01`.
4. **Description before explanation:** raw stage differences, sensitivity to products
   and dual-brand users made the shopper state and assortment impossible to ignore;
   `02` owns the current descriptive evidence, including the reach/conversion identity.
5. **A broader primary RQ:** the old stage comparison remained diagnostic; the recent
   direction distinguishes purchase-count shortfall, brand reach, browsing intensity
   and conversion. The selected `03` addresses corresponding adjusted associations.
6. **An appropriately limited decision:** patterns prioritise investigation and
   retailer-supported testing; no analysis here identifies broken pages, causal
   audience effects, optimal discounts or a budget allocation.

**Coverage boundary.** All substantive routes and their decision-relevant outcomes
are described here; aggregate supporting results are embedded. Current `01`/`02`
calculations are not copied into a second executable pipeline. Retired filenames
in the next section identify optional provenance only. Failed reasoning, outdated
measurement and provisional p-values are explained rather than silently promoted
to final findings. Reading this record requires no old code.

## 7. Consolidation and retirement of exploratory files

The following is a migration record, not a reading list. Every retained conceptual
lesson and its decision-relevant historical result is explained in this notebook; implemented measurement and descriptive
calculations are assigned to `01` and `02`. The three retired exploratory files
can be removed from the active project without breaking this sequence. Their
names and cell labels below document provenance only; reopening them is unnecessary.
Git history can preserve the original experiments.

| Source and blocks | Lesson preserved in 00 | Calculation owned elsewhere |
| --- | --- | --- |
| `Cleaning_Description`, 01–06 | Two different sources, schemas and missingness; reviews are usable for a different question | Behaviour sample counts: `02`, Table 3.1 |
| `Cleaning_Description`, 07–14 | Valid identifiers and a composite user/session key matter; no cross-file join | Event cleaning and reconciliation: `01`; measurement audit: `02`, Tables 3.3 and 3.6 |
| `Cleaning_Description`, 15–22 | Full category/brand coverage before selecting the scope; review coverage | TV activity, period and price figures: `02` |
| `Cleaning_Description`, 23–34 | Compare like units; shortlist by support, prices and concentration rather than gap significance | First-view table and timing: `01`; category gaps and TV prices: `02`, Sections 4.1–4.2 |
| `Cleaning_Description`, 35–39 and final Markdown | Satisfaction alternative, uncertainty about scope, and the need for browsing adjustment | Historical conclusions in Evidence E2–E4/E9; current RQ in Section 1 |
| `Edoardo_notebook`, 01–09 | Feasible first-view design, funnel denominators, price/month adjustment and repeated observations | Build: `01`; descriptive rates and prices: `02` |
| `Edoardo_notebook`, 10–11 | Popular products and selected dual-brand users can change the comparison | Concentration and funnel sensitivities: `02`; any additional modelling sensitivity later |
| `Edoardo_notebook`, 12–14 and final Markdown | Binary stage Logits and standardised predicted probabilities are methodological starting points | No historical coefficient or fitted probability is reused as a current result |
| `Shopper_Type_Analysis`, S01–S05 | Whole-period shopper labels can reveal composition but use future information | Current within-session shopper description: `02`, Sections 4.4–4.5 |
| `Shopper_Type_Analysis`, S06–S07 | Reweighting is descriptive; browsing more products changes the denominator | Current mix decomposition: `02`; no duplicate historical gap decomposition here |
| `Shopper_Type_Analysis`, S08–S10 | Compare models on a common sample; prior exposure is preferable to future shopper labels | Subsequent Logits; no historical numerical estimate is promoted to a final finding |
| `both_brands`, 1–5 | All-category and all-competitor checks; dual-brand restriction is selected | Cohort: `01`; category, price and mix results: `02` |
| `both_brands`, 6–7 | Pre-view covariates, conditional Logit feasibility, price-interaction sensitivity | Historical outcomes in Evidence E10; new models only for a stated concern |

The historical files used **strictly later** carts/purchases. Current `01` accepts
carts and purchases at the first-view timestamp; purchase-after-cart remains
strictly later. Therefore old numerical results are not silently combined with
current results. The old independent-row category z-statistics in `both_brands`
are not retained as evidence; `02` now treats repeated users and products.

**What is deliberately superseded, rather than promised as a future recovery:**
The fitted coefficients, whole-period shopper-type results, selected-user conditional
Logit and price-quintile experiment are preserved in the self-contained Evidence
record as historical findings. They are superseded as official estimates, not erased
from the decision history. Their purposes, actual outcomes and limits can be read
here without opening the original sources. A new model is justified by the current RQ and measurement
contract, not by preserving every discarded estimate. Old strict-timestamp
numbers and independent-row z-tests must not enter the current report by default.

## 8. Measurement choices motivated here, implemented only in 01

| Choice | Why it follows from the exploration | Implementation boundary |
| --- | --- | --- |
| User × session × product with a view | Views can repeat; session IDs are not guaranteed to identify a user; a product consideration is not a person or a unit sold | `01` constructs and checks the key; `00` does not recreate it |
| First view sets price and time | Predictors must describe the situation before the outcome | `01` saves `first_view`, `price` and `month` |
| Purchase in the same consideration/session | A clear observed window avoids inventing full customer histories | `01` constructs `purchased`; zero means no qualifying observed outcome |
| Same-second inclusion | Coarse timestamps cannot resolve every order; the current pipeline has an explicit rule | `01` audit and `02` measurement discussion; any alternative is a labelled later sensitivity |
| Retain all six TV brands for history | Other brands can be part of a shopper's path before LG or Samsung | `01` constructs `other_four` and the prior-brand flags |
| Whole-session prior browsing | History before the first view describes the ongoing search; restarting after purchases changes its meaning | Whole-session primary definition in `01`; reset sensitivity and results owned by `01`/`02` |
| Limit cross-session history claims | Whole sessions were sampled, not full user histories | Returning/prior-purchase flags describe recorded evidence, not new-customer or non-buyer status |
| Top models as a retrospective split | Concentration can make aggregate brand comparisons sensitive to the assortment | `01` defines and audits the rule; `02` reports results; `top_model` is not a first-view shopper characteristic |

Controls are justified by timing and the comparison they clarify, not selected just
because their raw conversion differences are large. Later cart entry, purchases,
`session_result`, eventual brand totals and whole-period shopper types must not be
used as if they were known at the first view in the main purchase model.

## 9. The analytical sequence after the scope decision

### 9.1 Distinguish purchase counts, reach and conversion

For brand *b*, let **S** be distinct user-sessions opening at least one model of that
brand, **C** the viewed considerations and **P** the purchased considerations. Then:

**P = S × (C / S) × (P / C)**.

This is an accounting identity: sessions reaching a brand, considerations per
reaching session, and purchase conversion. It motivates the organisation of `02`,
not a new model to fit in `00`. The factors and their contributions are calculated
only in `02`, Section 4.7. A session can reach both brands; reach counts are not
disjoint audiences, unique people or ad impressions. Purchased considerations
are not verified physical units, revenue or profit.

The identity separates a count difference from a rate difference. Its contributions
are descriptive accounting, not recoverable purchases, causal effects or recommended
budget shares. The prior decision to study TV/LG–Samsung is justified by the scope
evidence, not retrospectively by whichever decomposition share is largest.

### 9.2 Describe before estimating

`02` should establish the observed pattern and inspect prices, funnel denominators,
shopper state, navigation and concentration before substantive models are interpreted.
Its raw-gap regressions and uncertainty calculations are inferential support for
descriptive comparisons, not a completed adjusted explanation. This notebook makes
no claim that a particular gap persists, disappears or is equivalent to zero.

### 9.3 Proposed step 03: modelling the two sides of the question

The colleague's supplied `03_models.html`, headed **Models for Section 5: LG and
Samsung TVs**, makes the next step concrete. It follows `02` conceptually and reads
`tv_considerations.csv`, `tv_considerations_top_models.csv` and `tv_events_clean.csv`
from `01`. It does not need to rerun the scope screening or copy the descriptive
exhibits. The proposed executable notebook is not yet present in the repository.

| Block in proposed 03 | Unit and sample | Purpose and boundary |
| --- | --- | --- |
| **Model 1: conversion** | LG/Samsung considerations | Linear probability models with raw, position/price/week-adjusted and richer pre-view-history versions; top-model interaction; Gelbach decomposition of coefficient changes; targeted sensitivities, including purchase Logit average discrete contrasts. These are observed-covariate comparisons, not repeated observations of an identical shopper under alternative brands. |
| **Model 2: within-visit choice** | Opened TVs in visits with at least two considered models and exactly one purchased consideration; all six brands, plus an LG/Samsung head-to-head subset | Conditional Logit with visit fixed effects and a within-visit LPM check. This concerns which opened model was purchased conditional on the selected visit ending in one purchase, not unconditional conversion. |
| **Model 3: considerations per model** | One observation per viewed LG/Samsung product | Poisson conditional-mean regression with robust errors; alternative price controls, top-model exclusion, an observed-span offset and log-count OLS sensitivity. This concerns product-level consideration counts, not directly the brand-level number of sessions reaching any of its models. |
| **Model 4: optional price extension** | Considerations for conversion; product-day/week panels for counts | Product and time fixed effects examine price associations within a product. Prices on unobserved periods and the unobserved listing/availability window need explicit sensitivity checks; no causal elasticity or recommended discount follows. |

This records the submitted scope, not an endorsement of every implementation or
interpretation. The earlier purchase-Logit sequence was a methodological starting
point; it is not a separate pipeline to run in addition to `03`. The proposal uses
an LPM as the conversion baseline and Logit as a robustness check. A purchase Logit
still connects the binary outcome to the established course method. More advanced
blocks require a specific purpose and confirmation that they fit the course's
method requirements; no need to keep all extensions in the final report.

The baseline conversion controls in the supplied proposal are position, log price
and calendar-week effects. Its richer version replaces position with grouped prior
TV counts, rival/own-brand and other-four prior-view flags, plus returning visitor.
Keep the same sample when comparing specifications. Report counts, overlap, sparse
cells, convergence, LPM predictions outside [0,1] and uncertainty. `top_model` uses
the whole period and is a retrospective split, not a prospectively known shopper
characteristic; the blanket claim that all controls are known at first view needs
that exception.

Do not include redundant encodings of the same history. Brand-relative flags need
careful prediction design: a standardised profile must not combine inconsistent
absolute and relative histories. Differences in a coefficient or in a predicted
gap are conditional associations, not a causal explained share. The LPM/Gelbach
identity and the Logit probability contrast answer related but distinct questions.


### 9.4 Report probabilities with uncertainty, not coefficient changes alone

For the LPM baseline, a coefficient on brand gives a percentage-point difference
under its specified additive structure; inspect its probability-range limitations.
For the Logit checks, on a stated common population average the predicted probabilities for
the LG and Samsung profiles and report their paired difference in **percentage
points**. Report the odds ratio as a separate technical summary. In nonlinear
models, changes in log-odds coefficients across specifications are not an explained
share; even changes in standardised gaps are conditional associations.

The probability difference needs its **own** confidence interval from a justified
covariance/delta-method calculation or appropriate resampling. Coefficient intervals
are not probability-gap intervals. Earlier notebooks omitted gap intervals and
clustered only by user. Current `02` explicitly recognises user and product
dependence; later modelling should carry that concern forward with a documented
joint treatment, rather than silently returning to independent rows. Product-cluster
or small-cluster sensitivity is particularly important in restricted top-model
samples. The large number of considerations does not create many independent
brand assignments: brand is constant within product.

Do not add product fixed effects while expecting a separately identified brand
coefficient. No significant difference is not evidence of equivalence. Any
equivalence claim would require its own meaningful margin and appropriate test.

### 9.5 Secondary models and checks: one concern per addition

| Concern | Candidate check | Why it remains secondary |
| --- | --- | --- |
| Is the overall comparison sensitive to browsing history? | M2 versus M3 on the same population; alternative prior-view encodings | Incomplete observed history and remaining omitted variables prevent a causal explanation |
| Do the funnel diagnostics warrant a stage model? | Logit for `added_to_cart`; Logit for `purchased_after_cart` among eligible carts | Selected cart denominator; entry month should correspond to the relevant stage and may require rebuilding a cart timestamp not saved in the current CSV |
| Are aggregate differences sensitive to a few products? | Full versus prelabelled top/non-top or leave-one-product-out checks | Changes the assortment and independent product support; `02` owns descriptive split results |
| Could stable shopper differences matter? | Selected dual-brand users; conditional user Logit only with within-user outcome and covariate variation | Selected estimand; loses invariant outcomes; time-varying intent and brand-specific preferences remain |
| Is the price adjustment too rigid? | A prespecified nonlinear price form or a limited interaction with adequate overlap | Listed price is not randomly assigned; a price interaction is not elasticity or a discount recommendation |
| Does the observation rule matter? | Same-second, end-of-window and prior-purchase sensitivities | Changes what is observed or who is included; report changed denominators |
| Is the chosen link driving the average contrast? | A labelled Logit/LPM comparison on the same sample | Not a replacement for Logit diagnostics or dependence-aware uncertainty |

Whole-period cross-shopper profiles and their reweighting supplied the useful
warning that considering many models can reduce a per-consideration rate without
lowering a user's probability of any purchase. They do not demonstrate that shopper
mix causes the gap. Restricting to people who viewed both brands does not match
products, timing, intent or number of rows contributed by those people.

For clarity, the original descriptive labels mean **LG-only**, **Samsung-only** or
**both brands observed for that user over the full window**. They are not lifetime
loyalty segments. A user profile can count considered products, sessions, distinct
TV brands, other categories visited and whether any TV purchase was recorded;
none is a measured intention or demographic. Whole-period totals include future
information relative to earlier views. A shopper considering ten models and buying
one has one purchase across ten considerations but still one user with a purchase:
these denominators answer different questions.

The descriptive mix calculation has a precise interpretation. If `w_bg` is brand
`b`'s share of considerations in group `g` and `r_bg` is its within-group purchase
rate, its rate is `sum_g(w_bg * r_bg)`. Applying Samsung's weights to LG's group rates
gives `sum_g(w_Sg * r_Lg)`. Subtracting and adding this quantity splits the raw gap
into a composition term and a within-group term. The split depends on the reference
weights; it is not a causal attribution. For cart-to-purchase the weights must be
shares of eligible carts, not shares of all views. `02` owns the current descriptive
reweighting using first-view browsing state; there is no second implementation here.

The prospective alternative is **other brand observed strictly before this first
view**. Within-session history is primary; earlier-session exposure can miss
unsampled visits. A narrower observed history is not proof of an absence of earlier
exposure. A difference between retrospective and prior-only analyses diagnoses
definition sensitivity, not which visitor group an intervention can profitably target.

Separate funnel probabilities cannot generally be multiplied to reconstruct overall
purchase conversion because recorded purchases may lack an eligible cart path.
A persistent stage gap motivates investigation; it does not diagnose defective
pages, payment or delivery. Stop adding techniques once they no longer resolve a
stated uncertainty. PCA lacks a defensible measured-attribute set for this question;
multinomial choice modelling requires a defined choice outcome and choice set, not
the current priority-based consideration label.

## 10. Questions the rationale must withstand

| Likely question | Answer and evidence boundary |
| --- | --- |
| Why not use both supplied datasets? | Different products/users and no valid join; review feasibility is documented in Section 5. Two datasets are not intrinsically better than one coherent dataset. |
| Why TVs instead of washing machines? | A focused, supported two-brand comparison with a feasible benchmark; washing machines remain credible. `02`, Table 4.1 gives current support; no claim that TVs win every criterion. |
| Why Samsung? | The original all-competitor screen combined support, assortment and price proximity. Current TV price evidence is in `02`, Section 4.2. Similar prices do not equal comparable specifications. |
| Was scope chosen after looking at the data? | Yes. It is an exploratory, disclosed scope decision, not a preregistered test. All-category results cannot be used as independent confirmation of a category selected from them. |
| Why has the old funnel RQ evolved? | A purchase-count shortfall can reflect reach and browsing intensity as well as conversion. Funnel stages remain useful diagnostics rather than competing primary endpoints. |
| Why build 01 rather than use an old notebook's table? | One explicit current measurement contract, reproducible variables and audit trail; historical timing and history definitions differ. |
| Why is 02 needed before Logit? | It establishes the observed patterns, support and sensitivities a model must address; it prevents interpreting an adjusted coefficient without understanding its denominator. |
| Does a narrowing adjusted gap prove an audience problem? | No. It shows sensitivity to measured composition on a stated population; unobserved intent, specifications and selection remain. |
| Does first-brand order identify awareness or paid placement? | No. No ad exposure or placement is observed; even a random-order benchmark conditions on the models eventually viewed. |
| Can the findings tell LG where to spend its budget? | They can identify questions and candidates for a retailer-supported test. Returns and optimal allocation require incremental effects, costs and margins absent here. |
| Are non-top models equally good when a test is nonsignificant? | Not established. Weighting, product selection and uncertainty matter; no equivalence claim follows from a large p-value. |

The team's assessed report must present its own reasoning and disclose AI support.
This working notebook is not text to paste into the final report.

## 11. Reproducibility and handoff

The next cell records fingerprints of the maintained scripts, dictionary and raw
inputs, so this rationale can be checked against their local versions. It does not
read any retired notebook or script, fetch Git, read raw rows into
outputs, write datasets, regenerate results or assert that an independent variable
verifier exists. `tv_considerations_dictionary.md` mentions such a verifier; the
current repository must actually contain its code and log before verification can
be claimed. Read the scripts as the authority for implemented behaviour.

The revised report guide used for the RQ alignment is a local working document
(`Report_guide_Sections_1_to_4.pdf`, Section 2), not a runtime dependency. All rationale
needed to understand the route is included here. The maintained scripts are in the
repository; the original dictionary and CSVs remain in the sibling private data
directory. The three exploratory files are neither runtime nor reading dependencies.

Roadmap reference: the user-supplied `03_models.html` was inspected on 18 November
2026. It is an external review export, not a runtime dependency or proof that the
proposed notebook has been committed. Its SHA-256 is `d7d802efa82bf695e24b46c8a8f43c555b8969a1a1bd690e5bcdffc04405c27f`.


In [7]:
# 07. Record only maintained source versions, with no dependency on retired files.
source_paths = [
    Path('01_build_tv_dataset.py'), Path('02_preliminary_analysis.py'),
    Path('tv_considerations_dictionary.md'),
    *[DATA_DIR / name for name in required],
]
source_versions = []
for path in source_paths:
    if not path.is_file():
        raise FileNotFoundError(path)
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    source_versions.append({'Source': str(path), 'SHA-256': digest.hexdigest()})
display(pd.DataFrame(source_versions).set_index('Source'))


,SHA-256
Source,
01_build_tv_dataset.py,f688ead4ae5051a4012971b6beda5d707317d1db5d020f560e91ebf095873e67
02_preliminary_analysis.py,d92fdcf8107e4e1c86f0c3fd4334d20fe232ff18ef59531b35018838dde85acc
tv_considerations_dictionary.md,dc2d686f2f95d07720867cb4ff10c0784977328c13843c527819ce04b76fd296
../data/01_LG_Electronics_Appliances/electronics_market_behavior.csv,3303a56e62cff3e38491d58e41f40706317c500039c26f135bc89a811bc59022
../data/01_LG_Electronics_Appliances/appliance_brand_reviews.csv,64b57e87b44ec047403bf1b29bdc60ecd9ead2add64b8327ac6011524bb31a60
../data/01_LG_Electronics_Appliances/DATA_DICTIONARY.md,85d0b484b7a95abfcbdb49efbd78c21a9589470da9829d8850df8c6b52a3e8d6


**Handoff:** `00` explains the route and its limits. Run `01_build_tv_dataset.py`
to implement the measurement decisions, then `02_preliminary_analysis.py` to
produce the descriptive evidence. The proposed **03 models** notebook, supplied
for review as `03_models.html`, then addresses adjusted conversion, conditional
within-visit choice and product-level consideration counts, with price changes as
an optional extension. It reads the `01` datasets. Its estimates and conclusions
belong in that step, not in `00`. The HTML is not itself an executable notebook.
No old exploratory finding is a required result to recover.